In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Add_missing_heavy_atoms)=
# Add missing heavy atoms

*Adding missing non-hydrogen heavy atoms to a molecular system.*

Some molecular systems—especially those obtained from experimental techniques like X-ray crystallography—may lack certain heavy atoms due to resolution limits or local structural disorder. The function {func}`molsysmt.build.add_missing_heavy_atoms` reconstructs these missing non-hydrogen atoms whenever sufficient topological and geometrical template information is available. The native engine uses exact templates for MSE, SEP, and selected TPO gaps: it keeps their residue names and existing coordinates, adds atoms with the component's own elements and bonds, and warns when a repair is ambiguous or unsupported. MLY missing atoms remain unassessed because their placement is not yet validated. Inspect reconstructed coordinates before using them for simulation.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.build.add_missing_heavy_atoms`.
:::


## Basic usage

Let's show how this function works by building a peptide system and intentionally removing selected heavy atoms to simulate missing structural data:


In [2]:
import molsysmt as msm


In [3]:
molsys = msm.build.build_peptide('AceHisThrNme')


In [4]:
msm.info(molsys)


form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,43,4,1,1,1,1,1,1


First, let's remove hydrogen atoms from our system:


In [5]:
molsys = msm.remove(molsys, selection='atom_type=="H"')


In [6]:
msm.build.has_hydrogens(molsys)


False

Next, let's remove specific heavy atoms (`NE2`, `CD2`, `OG1`) to simulate experimental atom loss:


In [7]:
molsys = msm.remove(molsys, selection='atom_name in ["NE2", "CD2", "OG1"]')


We can verify which heavy atoms are detected as missing using {func}`molsysmt.build.get_missing_heavy_atoms`:


In [8]:
msm.build.get_missing_heavy_atoms(molsys)


{1: ['CD2', 'NE2'], 2: ['OG1']}

Now request bounded reconstruction. Restore visible diagnostics before the call so unassessed residues are apparent. A complete atom inventory and a validated conformation are different checks:


In [9]:
warnings.simplefilter("always")


In [10]:
molsys = msm.build.add_missing_heavy_atoms(molsys)


  return fn_to_wrap(**bound)



  native_out = append_atoms_to_molsys(



Inspect the atom inventory and query the remaining gaps. The native engine restores THR OG1 here. The HIS NE2/CD2 gap has no validated local placement and remains unassessed:


In [11]:
msm.info(molsys, element='atom')


index,id,name,type,group index,group id,group name,group type,component index,chain index,molecule index,molecule type,entity index,entity name
0,1,CH3,C,0,0,ACE,terminal capping,0,0,0,peptide,0,peptide 0
1,4,C,C,0,0,ACE,terminal capping,0,0,0,peptide,0,peptide 0
2,5,O,O,0,0,ACE,terminal capping,0,0,0,peptide,0,peptide 0
3,6,N,N,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0
4,8,CA,C,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0
5,10,CB,C,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0
6,13,CG,C,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0
7,14,ND1,N,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0
8,15,CE1,C,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0
9,21,C,C,1,1,HIS,amino acid,0,0,0,peptide,0,peptide 0


In [12]:
msm.build.get_missing_heavy_atoms(molsys)


{1: ['CD2', 'NE2']}

## Repair boundaries

Standard and curated modified residues share exact-name, element, connectivity
and local-anchor checks. Multiple missing side-chain atoms remain unassessed.
The native engine rejects nonfinite or collinear anchors and new-to-observed
bond lengths differing by more than 0.04 nm from their template. Placement is an
initial estimate; inspect clashes, stereo and environment before scientific use.
It does not establish an energy minimum. Hydrogens require a separate operation.

Native atom expansion requires one chemical state. Existing atom IDs, assignments,
state ID and provenance survive; chemical values for added atoms remain unknown
and connectivity completeness is partial. Atoms are sorted by group, so indices
can change. Named interaction analyses keep their definitions and remapped axes,
but their occurrences and evaluated coverage are cleared. Recalculate them.

Observed coordinates, time and box are retained. Existing B-factors, velocities
or force-field atom parameters cannot be extended without new values. The native
`attribute_policy='intersection'` default reports and drops unsupported values;
`attribute_policy='strict'` rejects the repair without changing your input.
PDBFixer is a separate optional reconstruction engine and supports only
intersection policy in this tool. Converting a native result back to another
form also follows that form's attribute capabilities.


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Build peptide <Tutorial_Build_peptide>`: Generate capped peptide structures from sequence with {func}`molsysmt.build.build_peptide`.
- {ref}`Info <Tutorial_Info>`: Display structural and topological summaries of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Remove <Tutorial_Remove>`: Remove atoms from a molecular system with {func}`molsysmt.basic.remove`.
- {ref}`Has hydrogens <Tutorial_Has_hydrogens>`: Check whether a molecular system contains hydrogen atoms with {func}`molsysmt.build.has_hydrogens`.
- {ref}`Get missing heavy atoms <Tutorial_Get_missing_heavy_atoms>`: Identify missing non-hydrogen heavy atoms with {func}`molsysmt.build.get_missing_heavy_atoms`.
- {ref}`Add missing heavy atoms <Tutorial_Add_missing_heavy_atoms>`: Reconstruct missing heavy atoms with {func}`molsysmt.build.add_missing_heavy_atoms`.
- {ref}`Add missing hydrogens <Tutorial_Add_missing_hydrogens>`: Add missing hydrogen atoms to a molecular system with {func}`molsysmt.build.add_missing_hydrogens`.
:::
